In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed/malware/Windows_Malware_Final_318_Features.csv


In [3]:
# ============================================================
# CELL 1 — Class-wise Feature Analysis
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load final dataset
# ------------------------------------------------------------

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed/malware/Windows_Malware_Final_318_Features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. Separate identifier, target and features
# ------------------------------------------------------------

TARGET = "Type"
ID_COL = "SHA256"

feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET]]

X = df[feature_cols]
y = df[TARGET]

print("ML features:", len(feature_cols))
print("Classes:", sorted(y.unique()))

# ------------------------------------------------------------
# 3. Identify feature groups
# ------------------------------------------------------------

api_features = [c for c in feature_cols
                if c in feature_cols[:200]]

# Use the known final construction:
# 200 API + 27 DLL + 46 PE Header + 45 PE Section

dll_features = feature_cols[200:227]
pe_header_features = feature_cols[227:273]
pe_section_features = feature_cols[273:318]

print("\nFeature groups:")
print("API        :", len(api_features))
print("DLL        :", len(dll_features))
print("PE Header  :", len(pe_header_features))
print("PE Section :", len(pe_section_features))

# ------------------------------------------------------------
# 4. Convert features to numeric
# ------------------------------------------------------------

X = X.apply(pd.to_numeric, errors="coerce")

# ------------------------------------------------------------
# 5. Calculate class-wise feature means
# ------------------------------------------------------------

class_means = {}

for cls in sorted(y.unique()):
    class_means[cls] = X[y == cls].mean()

class_means = pd.DataFrame(class_means)

print("\nClass-wise feature matrix:")
print(class_means.shape)

# ------------------------------------------------------------
# 6. Class-vs-rest discriminative score
# ------------------------------------------------------------

results = {}

for cls in sorted(y.unique()):

    cls_mask = y == cls

    cls_mean = X[cls_mask].mean()
    rest_mean = X[~cls_mask].mean()

    # Difference in feature prevalence/value
    difference = (cls_mean - rest_mean).abs()

    results[cls] = pd.DataFrame({
        "Feature": feature_cols,
        "Class_Mean": cls_mean.values,
        "Rest_Mean": rest_mean.values,
        "Absolute_Difference": difference.values
    }).sort_values(
        "Absolute_Difference",
        ascending=False
    )

# ------------------------------------------------------------
# 7. Display top 15 features for every class
# ------------------------------------------------------------

for cls in sorted(results):

    print("\n" + "=" * 70)
    print(f"TYPE {cls} — TOP 15 CLASS-DISCRIMINATIVE FEATURES")
    print("=" * 70)

    display(results[cls].head(15))

Dataset shape: (29489, 320)
ML features: 318
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Feature groups:
API        : 200
DLL        : 27
PE Header  : 46
PE Section : 45

Class-wise feature matrix:
(318, 7)

TYPE 0 — TOP 15 CLASS-DISCRIMINATIVE FEATURES


,Feature,Class_Mean,Rest_Mean,Absolute_Difference
297,idata_Characteristics,7.113815e+08,1.286191e+08,5.827624e+08
258,ImageBase,6.049173e+08,2.395443e+07,5.809629e+08
245,TimeDateStamp,1.142691e+09,1.697923e+09,5.552323e+08
292,bss_Characteristics,6.397034e+08,8.771096e+07,5.519925e+08
312,tls_Characteristics,5.913181e+08,9.533247e+07,4.959856e+08
317,pdata_Characteristics,2.899327e+08,6.107238e+06,2.838254e+08
307,reloc_Characteristics,8.465417e+08,5.811506e+08,2.653910e+08
282,data_Characteristics,1.447864e+09,1.643903e+09,1.960386e+08
277,text_Characteristics,1.605333e+09,1.453331e+09,1.520024e+08
302,rsrc_Characteristics,1.176732e+09,1.106631e+09,7.010076e+07



TYPE 1 — TOP 15 CLASS-DISCRIMINATIVE FEATURES


,Feature,Class_Mean,Rest_Mean,Absolute_Difference
282,data_Characteristics,3.099785e+09,1.330035e+09,1.769751e+09
307,reloc_Characteristics,1.486364e+08,6.902864e+08,5.416501e+08
245,TimeDateStamp,1.482149e+09,1.699617e+09,2.174689e+08
297,idata_Characteristics,4.543537e+07,1.904000e+08,1.449646e+08
277,text_Characteristics,1.575764e+09,1.439861e+09,1.359029e+08
292,bss_Characteristics,1.218954e+07,1.455586e+08,1.333691e+08
312,tls_Characteristics,4.425943e+07,1.438653e+08,9.960591e+07
258,ImageBase,7.401418e+06,7.192094e+07,6.451952e+07
287,rdata_Characteristics,3.300747e+08,3.724222e+08,4.234752e+07
317,pdata_Characteristics,1.282846e+06,2.887131e+07,2.758846e+07



TYPE 2 — TOP 15 CLASS-DISCRIMINATIVE FEATURES


...

,Feature,Class_Mean,Rest_Mean,Absolute_Difference
277,text_Characteristics,1.204432e+09,1.516769e+09,3.123372e+08
245,TimeDateStamp,1.902140e+09,1.612773e+09,2.893674e+08
282,data_Characteristics,1.834735e+09,1.589152e+09,2.455829e+08
307,reloc_Characteristics,4.369968e+08,6.315281e+08,1.945313e+08
287,rdata_Characteristics,2.359083e+08,3.920952e+08,1.561868e+08
302,rsrc_Characteristics,1.203043e+09,1.091975e+09,1.110679e+08
312,tls_Characteristics,1.777931e+08,1.163211e+08,6.147206e+07
297,idata_Characteristics,2.053970e+08,1.574611e+08,4.793583e+07
292,bss_Characteristics,1.581170e+08,1.155121e+08,4.260497e+07
258,ImageBase,2.880399e+07,6.761359e+07,3.880960e+07



TYPE 5 — TOP 15 CLASS-DISCRIMINATIVE FEATURES


,Feature,Class_Mean,Rest_Mean,Absolute_Difference
282,data_Characteristics,3.140765e+08,1.851243e+09,1.537167e+09
307,reloc_Characteristics,1.017661e+09,5.280236e+08,4.896377e+08
287,rdata_Characteristics,1.032493e+08,4.089224e+08,3.056731e+08
245,TimeDateStamp,1.907990e+09,1.621633e+09,2.863570e+08
297,idata_Characteristics,3.819333e+06,1.927266e+08,1.889073e+08
277,text_Characteristics,1.610613e+09,1.438375e+09,1.722375e+08
312,tls_Characteristics,2.291600e+06,1.476955e+08,1.454039e+08
292,bss_Characteristics,0.000000e+00,1.433444e+08,1.433444e+08
258,ImageBase,6.728855e+06,6.997800e+07,6.324915e+07
302,rsrc_Characteristics,1.072851e+09,1.117474e+09,4.462374e+07



TYPE 6 — TOP 15 CLASS-DISCRIMINATIVE FEATURES


,Feature,Class_Mean,Rest_Mean,Absolute_Difference
282,data_Characteristics,1.715549e+08,1.840810e+09,1.669255e+09
307,reloc_Characteristics,1.049086e+09,5.333509e+08,5.157354e+08
287,rdata_Characteristics,5.812836e+07,4.092545e+08,3.511261e+08
245,TimeDateStamp,1.904719e+09,1.627853e+09,2.768655e+08
297,idata_Characteristics,6.095858e+06,1.886059e+08,1.825100e+08
277,text_Characteristics,1.606694e+09,1.442397e+09,1.642972e+08
292,bss_Characteristics,1.741674e+06,1.402154e+08,1.384738e+08
312,tls_Characteristics,6.966695e+06,1.441045e+08,1.371378e+08
258,ImageBase,1.295823e+07,6.781416e+07,5.485592e+07
302,rsrc_Characteristics,1.075266e+09,1.116232e+09,4.096592e+07


In [4]:
# ============================================================
# CELL 2 — Feature Group Analysis by Malware Class
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Feature groups
# ------------------------------------------------------------

feature_groups = {
    "API": api_features,
    "DLL": dll_features,
    "PE_Header": pe_header_features,
    "PE_Section": pe_section_features
}

# ------------------------------------------------------------
# 2. Calculate class-vs-rest mean absolute difference
# ------------------------------------------------------------

group_results = []

for group_name, features in feature_groups.items():

    group_X = X[features]

    for cls in sorted(y.unique()):

        cls_mean = group_X[y == cls].mean()
        rest_mean = group_X[y != cls].mean()

        diff = (cls_mean - rest_mean).abs()

        group_results.append({
            "Group": group_name,
            "Type": int(cls),
            "Mean_Absolute_Difference": diff.mean(),
            "Median_Absolute_Difference": diff.median(),
            "Max_Absolute_Difference": diff.max()
        })

group_results = pd.DataFrame(group_results)

print("=" * 80)
print("FEATURE GROUP DISCRIMINATION")
print("=" * 80)

display(
    group_results.sort_values(
        ["Type", "Mean_Absolute_Difference"],
        ascending=[True, False]
    )
)

# ------------------------------------------------------------
# 3. Overall group statistics
# ------------------------------------------------------------

overall_group = (
    group_results
    .groupby("Group")
    [["Mean_Absolute_Difference",
      "Median_Absolute_Difference",
      "Max_Absolute_Difference"]]
    .mean()
    .sort_values(
        "Mean_Absolute_Difference",
        ascending=False
    )
)

print("\n" + "=" * 80)
print("OVERALL FEATURE GROUP COMPARISON")
print("=" * 80)

display(overall_group)

# ------------------------------------------------------------
# 4. Number of features with non-zero variation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FEATURE COUNTS BY GROUP")
print("=" * 80)

for group_name, features in feature_groups.items():

    group_X = X[features]

    nonzero = (group_X.nunique() > 1).sum()

    print(
        f"{group_name:12s}: "
        f"{len(features):3d} total | "
        f"{nonzero:3d} non-constant"
    )

FEATURE GROUP DISCRIMINATION


,Group,Type,Mean_Absolute_Difference,Median_Absolute_Difference,Max_Absolute_Difference
21,PE_Section,0,5.804142e+07,24981.260926,5.827624e+08
14,PE_Header,0,2.488840e+07,397.152844,5.809629e+08
0,API,0,1.064946e-01,0.102438,3.084603e-01
7,DLL,0,5.451977e-02,0.025483,2.158031e-01
22,PE_Section,1,6.495449e+07,23271.766611,1.769751e+09
15,PE_Header,1,6.408691e+06,459.258439,2.174689e+08
1,API,1,1.679926e-01,0.089902,4.871706e-01
8,DLL,1,7.145086e-02,0.042465,4.874567e-01
23,PE_Section,2,8.261836e+07,38891.698812,1.730858e+09
16,PE_Header,2,4.523987e+06,476.334138,1.695577e+08



OVERALL FEATURE GROUP COMPARISON


,Mean_Absolute_Difference,Median_Absolute_Difference,Max_Absolute_Difference
Group,,,
PE_Section,6.075311e+07,33289.222825,1.220382e+09
PE_Header,9.047877e+06,653.633119,2.809188e+08
API,1.517447e-01,0.121882,4.764700e-01
DLL,8.684520e-02,0.042219,4.292856e-01



FEATURE COUNTS BY GROUP
API         : 200 total | 200 non-constant
DLL         :  27 total |  27 non-constant
PE_Header   :  46 total |  46 non-constant
PE_Section  :  45 total |  45 non-constant


In [5]:
# ============================================================
# CELL 3 — Feature Sparsity & Frequency Analysis
# ============================================================

# ------------------------------------------------------------
# 1. Binary feature frequency
# ------------------------------------------------------------

binary_features = []

for col in feature_cols:
    unique_vals = X[col].dropna().unique()

    if set(unique_vals).issubset({0, 1}):
        binary_features.append(col)

print("Binary features:", len(binary_features))

binary_freq = pd.DataFrame({
    "Feature": binary_features,
    "Present_Count": X[binary_features].sum().values,
    "Present_Percentage": (
        X[binary_features].mean().values * 100
    )
})

# ------------------------------------------------------------
# 2. Most common binary features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 MOST COMMON BINARY FEATURES")
print("=" * 80)

display(
    binary_freq
    .sort_values("Present_Percentage", ascending=False)
    .head(20)
)

# ------------------------------------------------------------
# 3. Rarest binary features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 RAREST BINARY FEATURES")
print("=" * 80)

display(
    binary_freq
    .sort_values("Present_Percentage", ascending=True)
    .head(20)
)

# ------------------------------------------------------------
# 4. Sparsity buckets
# ------------------------------------------------------------

def sparsity_bucket(p):
    if p == 0:
        return "0%"
    elif p < 1:
        return "<1%"
    elif p < 5:
        return "1–5%"
    elif p < 10:
        return "5–10%"
    elif p < 25:
        return "10–25%"
    elif p < 50:
        return "25–50%"
    else:
        return ">=50%"

binary_freq["Frequency_Bucket"] = (
    binary_freq["Present_Percentage"]
    .apply(sparsity_bucket)
)

print("\n" + "=" * 80)
print("BINARY FEATURE FREQUENCY DISTRIBUTION")
print("=" * 80)

bucket_order = [
    "0%",
    "<1%",
    "1–5%",
    "5–10%",
    "10–25%",
    "25–50%",
    ">=50%"
]

bucket_counts = (
    binary_freq["Frequency_Bucket"]
    .value_counts()
    .reindex(bucket_order, fill_value=0)
)

display(bucket_counts.to_frame("Feature_Count"))

# ------------------------------------------------------------
# 5. Features present per sample
# ------------------------------------------------------------

presence_count = X[binary_features].sum(axis=1)

print("\n" + "=" * 80)
print("NUMBER OF BINARY FEATURES PRESENT PER SAMPLE")
print("=" * 80)

print(f"Mean   : {presence_count.mean():.2f}")
print(f"Median : {presence_count.median():.2f}")
print(f"Min    : {presence_count.min():.0f}")
print(f"Max    : {presence_count.max():.0f}")

# ------------------------------------------------------------
# 6. Group-wise binary sparsity
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GROUP-WISE BINARY SPARSITY")
print("=" * 80)

for group_name, features in feature_groups.items():

    group_binary = [
        f for f in features
        if f in binary_features
    ]

    if group_binary:
        mean_presence = X[group_binary].mean().mean() * 100

        print(
            f"{group_name:12s}: "
            f"{len(group_binary):3d} binary features | "
            f"mean presence = {mean_presence:.2f}%"
        )

Binary features: 227

TOP 20 MOST COMMON BINARY FEATURES


,Feature,Present_Count,Present_Percentage
201,kernel32.dll,14595,49.493031
2,getprocaddress,13454,45.623792
219,mscoree.dll,12367,41.937672
59,corexemain,12360,41.913934
23,exitprocess,11385,38.607616
205,user32.dll,11159,37.841229
18,getlasterror,10747,36.444098
56,loadlibrarya,10631,36.050731
45,getcurrentprocess,10542,35.748923
8,sleep,10183,34.531520



TOP 20 RAREST BINARY FEATURES


,Feature,Present_Count,Present_Percentage
216,ncrypt.dll,254,0.861338
223,msvcp90.dll,290,0.983418
215,msvcr90.dll,329,1.115670
220,imagehlp.dll,352,1.193665
226,shfolder.dll,420,1.424260
222,oleacc.dll,439,1.488691
221,oledlg.dll,482,1.634508
213,gdiplus.dll,656,2.224558
217,winspool.drv,668,2.265251
214,winhttp.dll,787,2.668792



BINARY FEATURE FREQUENCY DISTRIBUTION


,Feature_Count
Frequency_Bucket,
0%,0
<1%,2
1–5%,28
5–10%,60
10–25%,97
25–50%,40
>=50%,0



NUMBER OF BINARY FEATURES PRESENT PER SAMPLE
Mean   : 32.22
Median : 23.00
Min    : 0
Max    : 121

GROUP-WISE BINARY SPARSITY
API         : 200 binary features | mean presence = 14.69%
DLL         :  27 binary features | mean presence = 10.53%


In [6]:
# ============================================================
# CELL 4 — API & DLL Class Discrimination
# ============================================================

from sklearn.feature_selection import mutual_info_classif

# ------------------------------------------------------------
# 1. API Mutual Information
# ------------------------------------------------------------

print("=" * 80)
print("API FEATURE MUTUAL INFORMATION")
print("=" * 80)

api_mi = mutual_info_classif(
    X[api_features],
    y,
    discrete_features=True,
    random_state=42
)

api_mi_df = pd.DataFrame({
    "Feature": api_features,
    "Mutual_Information": api_mi,
    "Presence_Percentage": X[api_features].mean().values * 100
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(api_mi_df.head(30))

# ------------------------------------------------------------
# 2. DLL Mutual Information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DLL FEATURE MUTUAL INFORMATION")
print("=" * 80)

dll_mi = mutual_info_classif(
    X[dll_features],
    y,
    discrete_features=True,
    random_state=42
)

dll_mi_df = pd.DataFrame({
    "Feature": dll_features,
    "Mutual_Information": dll_mi,
    "Presence_Percentage": X[dll_features].mean().values * 100
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(dll_mi_df)

# ------------------------------------------------------------
# 3. Top features by group
# ------------------------------------------------------------

print("\nTop 15 APIs:")
display(api_mi_df.head(15))

print("\nTop 15 DLLs:")
display(dll_mi_df.head(15))

# ------------------------------------------------------------
# 4. Rare but informative features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RARE BUT INFORMATIVE APIs")
print("=" * 80)

display(
    api_mi_df[
        api_mi_df["Presence_Percentage"] < 5
    ]
    .sort_values("Mutual_Information", ascending=False)
    .head(20)
)

print("\n" + "=" * 80)
print("RARE BUT INFORMATIVE DLLs")
print("=" * 80)

display(
    dll_mi_df[
        dll_mi_df["Presence_Percentage"] < 5
    ]
    .sort_values("Mutual_Information", ascending=False)
    .head(20)
)

API FEATURE MUTUAL INFORMATION


,Feature,Mutual_Information,Presence_Percentage
59,corexemain,0.336290,41.913934
57,virtualalloc,0.184613,25.338262
35,heapalloc,0.182684,27.054156
2,getprocaddress,0.181030,45.623792
99,loadicona,0.179711,14.262946
56,loadlibrarya,0.176902,36.050731
1,setunhandledexceptionfilter,0.175531,28.295297
20,heapfree,0.174424,26.609922
46,terminateprocess,0.172364,28.688664
86,heapcreate,0.171360,21.696226



DLL FEATURE MUTUAL INFORMATION


,Feature,Mutual_Information,Presence_Percentage
19,mscoree.dll,0.336284,41.937672
1,kernel32.dll,0.220839,49.493031
24,msvbvm60.dll,0.155408,10.648038
5,user32.dll,0.152117,37.841229
10,gdi32.dll,0.116945,26.033436
11,winmm.dll,0.072751,5.771644
25,mfc42.dll,0.060232,3.523348
0,advapi32.dll,0.054233,22.466004
4,shlwapi.dll,0.052865,4.550849
6,msvcrt.dll,0.052655,6.985656



Top 15 APIs:


...

,Feature,Mutual_Information,Presence_Percentage
19,mscoree.dll,0.336284,41.937672
1,kernel32.dll,0.220839,49.493031
24,msvbvm60.dll,0.155408,10.648038
5,user32.dll,0.152117,37.841229
10,gdi32.dll,0.116945,26.033436
11,winmm.dll,0.072751,5.771644
25,mfc42.dll,0.060232,3.523348
0,advapi32.dll,0.054233,22.466004
4,shlwapi.dll,0.052865,4.550849
6,msvcrt.dll,0.052655,6.985656



RARE BUT INFORMATIVE APIs


,Feature,Mutual_Information,Presence_Percentage
132,isbadcodeptr,0.075601,4.126963
128,polyline,0.074842,4.828919
114,polygon,0.070508,4.818746
129,isbadwriteptr,0.067526,4.615280
137,playsounda,0.066507,3.591170
135,loadacceleratorsa,0.062555,3.299535
195,vbavartstne,0.062063,4.740751
136,translateacceleratora,0.061330,3.255451
106,movewindow,0.060305,4.476245
126,escape,0.056340,4.364339



RARE BUT INFORMATIVE DLLs


,Feature,Mutual_Information,Presence_Percentage
25,mfc42.dll,0.060232,3.523348
4,shlwapi.dll,0.052865,4.550849
17,winspool.drv,0.032082,2.265251
21,oledlg.dll,0.029382,1.634508
26,shfolder.dll,0.025441,1.424260
14,winhttp.dll,0.025303,2.668792
22,oleacc.dll,0.023289,1.488691
20,imagehlp.dll,0.021165,1.193665
13,gdiplus.dll,0.018736,2.224558
15,msvcr90.dll,0.017097,1.115670


In [7]:
# ============================================================
# CELL 5 — PE Header & PE Section Mutual Information
# ============================================================

from sklearn.feature_selection import mutual_info_classif

# ------------------------------------------------------------
# 1. PE Header MI
# ------------------------------------------------------------

print("=" * 80)
print("PE HEADER MUTUAL INFORMATION")
print("=" * 80)

header_mi = mutual_info_classif(
    X[pe_header_features],
    y,
    discrete_features=False,
    random_state=42
)

header_mi_df = pd.DataFrame({
    "Feature": pe_header_features,
    "Mutual_Information": header_mi
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(header_mi_df)

# ------------------------------------------------------------
# 2. PE Section MI
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PE SECTION MUTUAL INFORMATION")
print("=" * 80)

section_mi = mutual_info_classif(
    X[pe_section_features],
    y,
    discrete_features=False,
    random_state=42
)

section_mi_df = pd.DataFrame({
    "Feature": pe_section_features,
    "Mutual_Information": section_mi
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(section_mi_df)

# ------------------------------------------------------------
# 3. Top features from both groups
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 PE HEADER FEATURES")
print("=" * 80)

display(header_mi_df.head(20))

print("\n" + "=" * 80)
print("TOP 20 PE SECTION FEATURES")
print("=" * 80)

display(section_mi_df.head(20))

# ------------------------------------------------------------
# 4. MI distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MI DISTRIBUTION")
print("=" * 80)

for name, mi_df in [
    ("PE Header", header_mi_df),
    ("PE Section", section_mi_df)
]:

    print(f"\n{name}")

    print(f"  Features : {len(mi_df)}")
    print(f"  MI > 0   : {(mi_df['Mutual_Information'] > 0).sum()}")
    print(f"  MI >= .01: {(mi_df['Mutual_Information'] >= 0.01).sum()}")
    print(f"  MI >= .05: {(mi_df['Mutual_Information'] >= 0.05).sum()}")
    print(f"  MI >= .10: {(mi_df['Mutual_Information'] >= 0.10).sum()}")

PE HEADER MUTUAL INFORMATION


,Feature,Mutual_Information
29,AddressOfEntryPoint,1.090807
18,TimeDateStamp,1.083475
27,SizeOfInitializedData,0.938969
26,SizeOfCode,0.917340
40,SizeOfImage,0.767965
42,CheckSum,0.658241
24,MajorLinkerVersion,0.570699
15,e_lfanew,0.549234
22,Characteristics,0.517962
44,DllCharacteristics,0.513375



PE SECTION MUTUAL INFORMATION


,Feature,Mutual_Information
25,rsrc_Misc_VirtualSize,0.988498
0,text_Misc_VirtualSize,0.980582
5,data_Misc_VirtualSize,0.916409
28,rsrc_PointerToRawData,0.892533
2,text_SizeOfRawData,0.885003
26,rsrc_VirtualAddress,0.788783
8,data_PointerToRawData,0.779425
6,data_VirtualAddress,0.701596
27,rsrc_SizeOfRawData,0.687709
7,data_SizeOfRawData,0.679569



TOP 20 PE HEADER FEATURES


,Feature,Mutual_Information
29,AddressOfEntryPoint,1.090807
18,TimeDateStamp,1.083475
27,SizeOfInitializedData,0.938969
26,SizeOfCode,0.917340
40,SizeOfImage,0.767965
42,CheckSum,0.658241
24,MajorLinkerVersion,0.570699
15,e_lfanew,0.549234
22,Characteristics,0.517962
44,DllCharacteristics,0.513375



TOP 20 PE SECTION FEATURES


,Feature,Mutual_Information
25,rsrc_Misc_VirtualSize,0.988498
0,text_Misc_VirtualSize,0.980582
5,data_Misc_VirtualSize,0.916409
28,rsrc_PointerToRawData,0.892533
2,text_SizeOfRawData,0.885003
26,rsrc_VirtualAddress,0.788783
8,data_PointerToRawData,0.779425
6,data_VirtualAddress,0.701596
27,rsrc_SizeOfRawData,0.687709
7,data_SizeOfRawData,0.679569



MI DISTRIBUTION

PE Header
  Features : 46
  MI > 0   : 46
  MI >= .01: 46
  MI >= .05: 27
  MI >= .10: 20

PE Section
  Features : 45
  MI > 0   : 45
  MI >= .01: 44
  MI >= .05: 39
  MI >= .10: 25


In [8]:
# ============================================================
# CELL 6 — Numerical PE Feature Distribution Analysis
# ============================================================

numerical_features = pe_header_features + pe_section_features

numeric_stats = pd.DataFrame({
    "Feature": numerical_features,
    "Min": X[numerical_features].min().values,
    "Max": X[numerical_features].max().values,
    "Mean": X[numerical_features].mean().values,
    "Median": X[numerical_features].median().values,
    "Std": X[numerical_features].std().values,
    "Unique_Values": X[numerical_features].nunique().values
})

# ------------------------------------------------------------
# 1. Features with largest ranges
# ------------------------------------------------------------

numeric_stats["Range"] = (
    numeric_stats["Max"] -
    numeric_stats["Min"]
)

print("=" * 80)
print("NUMERICAL FEATURES WITH LARGEST RANGES")
print("=" * 80)

display(
    numeric_stats
    .sort_values("Range", ascending=False)
    .head(20)
)

# ------------------------------------------------------------
# 2. Features with smallest ranges
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERICAL FEATURES WITH SMALLEST RANGES")
print("=" * 80)

display(
    numeric_stats
    .sort_values("Range", ascending=True)
    .head(20)
)

# ------------------------------------------------------------
# 3. Strongly skewed features
# ------------------------------------------------------------

numeric_stats["Skewness"] = (
    X[numerical_features]
    .skew()
    .values
)

print("\n" + "=" * 80)
print("MOST SKEWED NUMERICAL FEATURES")
print("=" * 80)

display(
    numeric_stats
    .sort_values(
        "Skewness",
        key=lambda s: s.abs(),
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERICAL FEATURE SUMMARY")
print("=" * 80)

print("Total numerical features :", len(numerical_features))
print("Features with max > 1e9   :", (numeric_stats["Max"] > 1e9).sum())
print("Features with max > 1e6   :", (numeric_stats["Max"] > 1e6).sum())
print("Features with max > 1e3   :", (numeric_stats["Max"] > 1e3).sum())
print("Features with max <= 1    :", (numeric_stats["Max"] <= 1).sum())

NUMERICAL FEATURES WITH LARGEST RANGES


,Feature,Min,Max,Mean,Median,Std,Unique_Values,Range
31,ImageBase,0,6442450944,6.093322e+07,4.194304e+06,5.451183e+08,26,6442450944
26,SizeOfCode,0,4294967295,1.394187e+06,2.048000e+05,6.616386e+07,3087,4294967295
18,TimeDateStamp,0,4294967295,1.662582e+09,1.609344e+09,6.443323e+08,20139,4294967295
27,SizeOfInitializedData,0,4294967295,1.967538e+06,5.632000e+04,5.623653e+07,2432,4294967295
28,SizeOfUninitializedData,0,4294967295,2.378721e+05,0.000000e+00,2.514097e+07,110,4294967295
75,rsrc_Characteristics,0,4026531904,1.111093e+09,1.073742e+09,6.552939e+08,10,4026531904
42,CheckSum,0,3885444380,1.114906e+06,0.000000e+00,4.224476e+07,10839,3885444380
80,reloc_Characteristics,0,3791650880,5.980430e+08,1.107296e+09,6.172093e+08,11,3791650880
50,text_Characteristics,0,3763339296,1.463006e+09,1.610613e+09,4.935709e+08,16,3763339296
60,rdata_Characteristics,0,3758096480,3.652104e+08,0.000000e+00,6.569478e+08,18,3758096480



NUMERICAL FEATURES WITH SMALLEST RANGES


,Feature,Min,Max,Mean,Median,Std,Unique_Values,Range
43,Subsystem,2,3,2.078572,2.0,0.269074,2,1
35,MinorOperatingSystemVersion,0,3,0.088236,0.0,0.332822,4,3
38,MajorSubsystemVersion,3,10,4.404490,4.0,0.738294,6,7
34,MajorOperatingSystemVersion,1,10,4.328631,4.0,0.764715,7,9
39,MinorSubsystemVersion,0,10,0.098850,0.0,0.353690,4,10
21,SizeOfOptionalHeader,224,240,224.334226,224.0,2.288251,2,16
17,NumberOfSections,2,49,4.131168,3.0,2.091947,21,47
24,MajorLinkerVersion,0,252,20.190512,9.0,23.076896,97,252
25,MinorLinkerVersion,0,255,3.448981,0.0,10.024095,34,255
23,Magic,267,523,272.347621,267.0,36.612012,2,256



MOST SKEWED NUMERICAL FEATURES


,Feature,Min,Max,Mean,Median,Std,Unique_Values,Range,Skewness
37,MinorImageVersion,0,4518,4.268032e-01,0.0,2.634276e+01,14,4518,171.049037
28,SizeOfUninitializedData,0,4294967295,2.378721e+05,0.0,2.514097e+07,110,4294967295,169.093175
29,AddressOfEntryPoint,4096,2025880815,4.915365e+05,54880.0,1.209462e+07,14004,2025876719,159.553826
86,pdata_Misc_VirtualSize,0,3284992,3.664328e+02,0.0,2.020330e+04,301,3284992,147.000827
66,idata_Misc_VirtualSize,0,3170304,7.176365e+02,0.0,1.979666e+04,684,3170304,140.645880
61,bss_Misc_VirtualSize,0,7626448,1.474914e+03,0.0,5.108714e+04,248,7626448,126.399780
81,tls_Misc_VirtualSize,0,1425408,2.447210e+02,0.0,9.716558e+03,44,1425408,112.758465
88,pdata_SizeOfRawData,0,509025,1.268816e+02,0.0,3.551936e+03,61,509025,106.650697
49,text_PointerToRawData,0,8022528,4.490804e+03,1024.0,6.015170e+04,181,8022528,85.619255
82,tls_VirtualAddress,0,49324032,2.882659e+04,0.0,4.010588e+05,272,49324032,82.800929



NUMERICAL FEATURE SUMMARY
Total numerical features : 91
Features with max > 1e9   : 18
Features with max > 1e6   : 53
Features with max > 1e3   : 80
Features with max <= 1    : 0


In [10]:
# ============================================================
# CELL 7 — LLM Input Schema Prototype
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Load the LOCKED training split
# ------------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/mohdtahasayed/malware-2-0/Malware_Train_60.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("=" * 80)
print("TRAINING DATA")
print("=" * 80)
print("Shape:", train_df.shape)

# ------------------------------------------------------------
# 2. Feature groups
# ------------------------------------------------------------

api_features = [c for c in train_df.columns if c in api_features]
dll_features = [c for c in train_df.columns if c in dll_features]
header_features = [c for c in train_df.columns if c in pe_header_features]
section_features = [c for c in train_df.columns if c in pe_section_features]

print("\nFeature groups:")
print("API       :", len(api_features))
print("DLL       :", len(dll_features))
print("PE Header :", len(header_features))
print("PE Section:", len(section_features))

# ------------------------------------------------------------
# 3. Malware class mapping
# ------------------------------------------------------------

CLASS_NAMES = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

# ------------------------------------------------------------
# 4. Convert one row into compact structured input
# ------------------------------------------------------------

def build_llm_input(row):
    
    # Only list binary API features that are PRESENT
    present_apis = [
        feature
        for feature in api_features
        if row[feature] == 1
    ]

    # Only list binary DLL features that are PRESENT
    present_dlls = [
        feature
        for feature in dll_features
        if row[feature] == 1
    ]

    # Preserve numerical PE Header values
    pe_header = {
        feature: row[feature]
        for feature in header_features
    }

    # Preserve numerical PE Section values
    pe_section = {
        feature: row[feature]
        for feature in section_features
    }

    # Convert NumPy values into normal Python values
    def clean_value(value):
        if pd.isna(value):
            return None

        if isinstance(value, (np.integer,)):
            return int(value)

        if isinstance(value, (np.floating,)):
            return float(value)

        return value

    pe_header = {
        k: clean_value(v)
        for k, v in pe_header.items()
    }

    pe_section = {
        k: clean_value(v)
        for k, v in pe_section.items()
    }

    # --------------------------------------------------------
    # Structured prompt
    # --------------------------------------------------------

    prompt = f"""Analyze the following Windows PE static feature profile.

[API FUNCTIONS PRESENT]
{", ".join(present_apis) if present_apis else "None"}

[DLLS IMPORTED]
{", ".join(present_dlls) if present_dlls else "None"}

[PE HEADER]
{json.dumps(pe_header, separators=(",", ":"))}

[PE SECTIONS]
{json.dumps(pe_section, separators=(",", ":"))}

Identify the malware family represented by this static profile.
Base the assessment only on the provided static features.
Do not assume runtime behavior that is not supported by the input.
"""

    return prompt


# ------------------------------------------------------------
# 5. Build 10 prototype examples
# ------------------------------------------------------------

sample_df = train_df.sample(
    n=10,
    random_state=42
).reset_index(drop=True)

prototype_records = []

for _, row in sample_df.iterrows():

    label = int(row["Type"])

    record = {
        "id": row["SHA256"],
        "input": build_llm_input(row),
        "target": CLASS_NAMES[label],
        "target_id": label
    }

    prototype_records.append(record)


# ------------------------------------------------------------
# 6. Display examples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("10 SAMPLE LLM RECORDS")
print("=" * 80)

for i, record in enumerate(prototype_records, 1):

    print(f"\n{'=' * 30} SAMPLE {i} {'=' * 30}")
    print("SHA256 :", record["id"])
    print("TARGET :", record["target"])
    print("TARGET ID:", record["target_id"])
    print("\nINPUT:")
    print(record["input"])


# ------------------------------------------------------------
# 7. Approximate input size
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INPUT SIZE ANALYSIS")
print("=" * 80)

for i, record in enumerate(prototype_records, 1):

    # Rough character-based estimate.
    # Actual tokenizer analysis will come later after selecting
    # the final model/tokenizer.
    chars = len(record["input"])
    approx_tokens = chars / 4

    print(
        f"Sample {i:2d}: "
        f"{chars:6d} chars | "
        f"~{approx_tokens:6.0f} estimated tokens | "
        f"target={record['target']}"
    )


# ------------------------------------------------------------
# 8. Save prototype JSONL
# ------------------------------------------------------------

prototype_path = "/kaggle/working/LLM_Schema_Prototype_10.jsonl"

with open(prototype_path, "w", encoding="utf-8") as f:
    for record in prototype_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nSaved:")
print(prototype_path)

TRAINING DATA
Shape: (17693, 320)

Feature groups:
API       : 200
DLL       : 27
PE Header : 46
PE Section: 45

10 SAMPLE LLM RECORDS

============================== SAMPLE 1 ==============================
SHA256 : 2df7b0d6ed9d2ad1dba3a9ba6b4d44bc4d7667b25b78b02b81cb0fa5b04fb9dd
TARGET : Benign
TARGET ID: 0

INPUT:
Analyze the following Windows PE static feature profile.

[API FUNCTIONS PRESENT]
setunhandledexceptionfilter, getcurrentprocessid, sleep, gettickcount, getlasterror, getcurrentthreadid, deletecriticalsection, queryperformancecounter, getsystemtimeasfiletime, entercriticalsection, leavecriticalsection, unhandledexceptionfilter, getcurrentprocess, terminateprocess, initializecriticalsection, exit, getstartupinfoa, onexit, dllonexit, initterm, setusermatherr, setapptype, tlsgetvalue

[DLLS IMPORTED]
kernel32.dll, msvcrt.dll

[PE HEADER]
{"e_cblp":144,"e_cp":3,"e_crlc":0,"e_cparhdr":4,"e_minalloc":0,"e_maxalloc":65535,"e_ss":0,"e_sp":184,"e_csum":0,"e_ip":0,"e_cs":0,"e_lfarlc"

In [12]:
# ============================================================
# CELL 8 — Deterministic Per-Sample Evidence Extraction
# ============================================================

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Load LOCKED training data
# ------------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/mohdtahasayed/malware-2-0/Malware_Train_60.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("=" * 80)
print("TRAINING DATA")
print("=" * 80)
print("Shape:", train_df.shape)


# ------------------------------------------------------------
# 2. Feature groups
# ------------------------------------------------------------

api_features = [c for c in train_df.columns if c in api_features]
dll_features = [c for c in train_df.columns if c in dll_features]
header_features = [c for c in train_df.columns if c in pe_header_features]
section_features = [c for c in train_df.columns if c in pe_section_features]

feature_groups = {
    "API": api_features,
    "DLL": dll_features,
    "PE_HEADER": header_features,
    "PE_SECTION": section_features
}

print("\nFeature groups:")
for group, features in feature_groups.items():
    print(f"{group:12s}: {len(features)}")


# ------------------------------------------------------------
# 3. Class mapping
# ------------------------------------------------------------

CLASS_NAMES = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}


# ------------------------------------------------------------
# 4. Calculate training-set statistics
#
# IMPORTANT:
# Evidence statistics are calculated ONLY from TRAINING data.
# Validation and test data are never used here.
# ------------------------------------------------------------

y_train = train_df["Type"].astype(int)

class_means = {}
overall_means = {}
overall_stds = {}

for group, features in feature_groups.items():

    overall_means[group] = train_df[features].mean()

    overall_stds[group] = (
        train_df[features]
        .std()
        .replace(0, np.nan)
    )

    class_means[group] = (
        train_df
        .groupby("Type")[features]
        .mean()
    )


# ------------------------------------------------------------
# 5. Calculate class-vs-rest standardized differences
# ------------------------------------------------------------

class_effects = {}

for group, features in feature_groups.items():

    group_effects = {}

    for class_id in CLASS_NAMES:

        class_mean = class_means[group].loc[class_id]

        rest_count = len(train_df) - (y_train == class_id).sum()

        rest_sum = (
            train_df[features].sum()
            - class_mean * (y_train == class_id).sum()
        )

        rest_mean = rest_sum / rest_count

        # Standardized class-vs-rest difference
        effect = (
            (class_mean - rest_mean)
            / overall_stds[group]
        )

        effect = effect.replace(
            [np.inf, -np.inf],
            np.nan
        ).fillna(0)

        group_effects[class_id] = effect

    class_effects[group] = group_effects


# ------------------------------------------------------------
# 6. Deterministic evidence extractor
# ------------------------------------------------------------

def clean_value(value):

    if pd.isna(value):
        return None

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    return value


def extract_sample_evidence(row, class_id, top_k=3):

    evidence = {}

    for group, features in feature_groups.items():

        effects = class_effects[group][class_id]

        # ----------------------------------------------------
        # Binary API / DLL:
        # Only features PRESENT in the sample are candidates.
        # ----------------------------------------------------

        if group in ["API", "DLL"]:

            present = [
                feature
                for feature in features
                if row[feature] == 1
            ]

            if not present:
                evidence[group] = []
                continue

            candidates = effects.loc[present]

            # Select features with strongest class-specific
            # standardized difference.
            selected = (
                candidates.abs()
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = [
                {
                    "feature": feature,
                    "value": 1,
                    "class_effect": round(
                        float(effects[feature]), 4
                    )
                }
                for feature in selected
            ]

        # ----------------------------------------------------
        # Numerical PE features:
        # Select features with strongest class-specific effect
        # and report their actual sample values.
        # ----------------------------------------------------

        else:

            candidates = effects.loc[features]

            selected = (
                candidates.abs()
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = [
                {
                    "feature": feature,
                    "value": clean_value(row[feature]),
                    "class_effect": round(
                        float(effects[feature]), 4
                    )
                }
                for feature in selected
            ]

    return evidence


# ------------------------------------------------------------
# 7. Convert evidence into human-readable grounded text
# ------------------------------------------------------------

def evidence_to_text(evidence):

    lines = []

    for group in ["API", "DLL", "PE_HEADER", "PE_SECTION"]:

        items = evidence[group]

        if not items:
            continue

        lines.append(f"{group}:")

        for item in items:

            feature = item["feature"]
            value = item["value"]

            if group in ["API", "DLL"]:
                lines.append(
                    f"- {feature} is present."
                )
            else:
                lines.append(
                    f"- {feature} = {value}."
                )

    return "\n".join(lines)


# ------------------------------------------------------------
# 8. Generate 10 prototype evidence records
# ------------------------------------------------------------

sample_df = (
    train_df
    .sample(n=10, random_state=42)
    .reset_index(drop=True)
)

prototype_evidence = []

for _, row in sample_df.iterrows():

    class_id = int(row["Type"])

    evidence = extract_sample_evidence(
        row,
        class_id,
        top_k=3
    )

    evidence_text = evidence_to_text(evidence)

    record = {
        "id": row["SHA256"],
        "target": CLASS_NAMES[class_id],
        "target_id": class_id,
        "evidence": evidence,
        "evidence_text": evidence_text
    }

    prototype_evidence.append(record)


# ------------------------------------------------------------
# 9. Display
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("10 DETERMINISTIC EVIDENCE EXAMPLES")
print("=" * 80)

for i, record in enumerate(prototype_evidence, 1):

    print(f"\n{'=' * 30} SAMPLE {i} {'=' * 30}")

    print("SHA256 :", record["id"])
    print("TARGET :", record["target"])

    print("\nEVIDENCE:")
    print(record["evidence_text"])


# ------------------------------------------------------------
# 10. Save prototype
# ------------------------------------------------------------

output_path = "/kaggle/working/LLM_Evidence_Prototype_10.jsonl"

with open(output_path, "w", encoding="utf-8") as f:

    for record in prototype_evidence:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("\nSaved:")
print(output_path)

TRAINING DATA
Shape: (17693, 320)

Feature groups:
API         : 200
DLL         : 27
PE_HEADER   : 46
PE_SECTION  : 45

10 DETERMINISTIC EVIDENCE EXAMPLES

============================== SAMPLE 1 ==============================
SHA256 : 2df7b0d6ed9d2ad1dba3a9ba6b4d44bc4d7667b25b78b02b81cb0fa5b04fb9dd
TARGET : Benign

EVIDENCE:
API:
- exit is present.
- initterm is present.
- setusermatherr is present.
DLL:
- msvcrt.dll is present.
- kernel32.dll is present.
PE_HEADER:
- Magic = 523.
- SizeOfOptionalHeader = 240.
- Machine = 34404.
PE_SECTION:
- pdata_Characteristics = 1076887616.
- bss_Characteristics = 3228565632.
- idata_Characteristics = 3224371264.

============================== SAMPLE 2 ==============================
SHA256 : 2dd0bfc0d560531b164d18f09e870fdb9b4985527813d7358914218546f83b45
TARGET : BankingTrojan

EVIDENCE:
API:
- raiseexception is present.
- loadlibrarya is present.
- getprocaddress is present.
DLL:
- kernel32.dll is present.
- oleaut32.dll is present.
- user32.d

In [14]:
# ============================================================
# CELL 8B — Sample-Specific Deterministic Evidence
# ============================================================

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Load LOCKED training data
# ------------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/mohdtahasayed/malware-2-0/Malware_Train_60.csv"

train_df = pd.read_csv(TRAIN_PATH)

# ------------------------------------------------------------
# 2. Feature groups
# ------------------------------------------------------------

api_features = [c for c in train_df.columns if c in api_features]
dll_features = [c for c in train_df.columns if c in dll_features]
header_features = [c for c in train_df.columns if c in pe_header_features]
section_features = [c for c in train_df.columns if c in pe_section_features]

feature_groups = {
    "API": api_features,
    "DLL": dll_features,
    "PE_HEADER": header_features,
    "PE_SECTION": section_features
}

# ------------------------------------------------------------
# 3. Class mapping
# ------------------------------------------------------------

CLASS_NAMES = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

# ------------------------------------------------------------
# 4. Training statistics
# ------------------------------------------------------------

y_train = train_df["Type"].astype(int)

class_counts = y_train.value_counts().to_dict()

class_means = {}
rest_means = {}
feature_stds = {}
class_effects = {}

for group, features in feature_groups.items():

    class_means[group] = (
        train_df
        .groupby("Type")[features]
        .mean()
    )

    feature_stds[group] = (
        train_df[features]
        .std()
        .replace(0, np.nan)
    )

    class_effects[group] = {}

    for class_id in CLASS_NAMES:

        n_class = class_counts[class_id]

        class_mean = class_means[group].loc[class_id]

        rest_sum = (
            train_df[features].sum()
            - class_mean * n_class
        )

        rest_count = len(train_df) - n_class

        rest_mean = rest_sum / rest_count

        rest_means[group] = rest_means.get(group, {})
        rest_means[group][class_id] = rest_mean

        effect = (
            (class_mean - rest_mean)
            / feature_stds[group]
        )

        effect = (
            effect
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )

        class_effects[group][class_id] = effect


# ------------------------------------------------------------
# 5. Sample-specific evidence extraction
# ------------------------------------------------------------

def extract_sample_evidence(row, class_id, top_k=3):

    evidence = {}

    for group, features in feature_groups.items():

        effects = class_effects[group][class_id]

        # ====================================================
        # API / DLL
        # ====================================================

        if group in ["API", "DLL"]:

            present = [
                feature
                for feature in features
                if row[feature] == 1
            ]

            if not present:
                evidence[group] = []
                continue

            # For binary features:
            # class effect itself captures how strongly
            # presence is associated with the class.
            scores = effects.loc[present].abs()

            selected = (
                scores
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = [
                {
                    "feature": feature,
                    "value": 1,
                    "evidence_score": round(
                        float(scores[feature]), 4
                    )
                }
                for feature in selected
            ]

        # ====================================================
        # PE HEADER / PE SECTION
        # ====================================================

        else:

            rest_mean = rest_means[group][class_id]
            std = feature_stds[group]

            sample_values = row[features].astype(float)

            # Sample-specific standardized deviation
            sample_deviation = (
                (sample_values - rest_mean)
                / std
            )

            sample_deviation = (
                sample_deviation
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0)
            )

            # Combine:
            #   class-specific effect
            #   sample-specific deviation
            #
            # This rewards features that are both:
            #   1. class-discriminative
            #   2. distinctive for this sample

            scores = (
                effects.abs()
                * sample_deviation.abs()
            )

            selected = (
                scores
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = [
                {
                    "feature": feature,
                    "value": (
                        int(row[feature])
                        if float(row[feature]).is_integer()
                        else float(row[feature])
                    ),
                    "evidence_score": round(
                        float(scores[feature]), 4
                    )
                }
                for feature in selected
            ]

    return evidence


# ------------------------------------------------------------
# 6. Convert evidence to grounded text
# ------------------------------------------------------------

def evidence_to_text(evidence):

    lines = []

    for group in [
        "API",
        "DLL",
        "PE_HEADER",
        "PE_SECTION"
    ]:

        items = evidence[group]

        if not items:
            continue

        lines.append(f"{group}:")

        for item in items:

            feature = item["feature"]
            value = item["value"]

            if group in ["API", "DLL"]:
                lines.append(
                    f"- {feature} is present."
                )
            else:
                lines.append(
                    f"- {feature} = {value}."
                )

    return "\n".join(lines)


# ------------------------------------------------------------
# 7. Generate 10 prototype examples
# ------------------------------------------------------------

sample_df = (
    train_df
    .sample(n=10, random_state=42)
    .reset_index(drop=True)
)

prototype_records = []

for _, row in sample_df.iterrows():

    class_id = int(row["Type"])

    evidence = extract_sample_evidence(
        row,
        class_id,
        top_k=3
    )

    prototype_records.append({
        "id": row["SHA256"],
        "target": CLASS_NAMES[class_id],
        "target_id": class_id,
        "evidence": evidence,
        "evidence_text": evidence_to_text(evidence)
    })


# ------------------------------------------------------------
# 8. Display
# ------------------------------------------------------------

print("=" * 80)
print("SAMPLE-SPECIFIC DETERMINISTIC EVIDENCE")
print("=" * 80)

for i, record in enumerate(prototype_records, 1):

    print(f"\n{'=' * 30} SAMPLE {i} {'=' * 30}")

    print("SHA256 :", record["id"])
    print("TARGET :", record["target"])

    print("\nEVIDENCE:")
    print(record["evidence_text"])


# ------------------------------------------------------------
# 9. Save prototype
# ------------------------------------------------------------

output_path = "/kaggle/working/LLM_Evidence_Prototype_10_v2.jsonl"

with open(output_path, "w", encoding="utf-8") as f:

    for record in prototype_records:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("\nSaved:")
print(output_path)

SAMPLE-SPECIFIC DETERMINISTIC EVIDENCE

============================== SAMPLE 1 ==============================
SHA256 : 2df7b0d6ed9d2ad1dba3a9ba6b4d44bc4d7667b25b78b02b81cb0fa5b04fb9dd
TARGET : Benign

EVIDENCE:
API:
- exit is present.
- initterm is present.
- setusermatherr is present.
DLL:
- msvcrt.dll is present.
- kernel32.dll is present.
PE_HEADER:
- Magic = 523.
- SizeOfOptionalHeader = 240.
- Machine = 34404.
PE_SECTION:
- pdata_Characteristics = 1076887616.
- bss_Characteristics = 3228565632.
- tls_Characteristics = 3227516992.

============================== SAMPLE 2 ==============================
SHA256 : 2dd0bfc0d560531b164d18f09e870fdb9b4985527813d7358914218546f83b45
TARGET : BankingTrojan

EVIDENCE:
API:
- raiseexception is present.
- loadlibrarya is present.
- getprocaddress is present.
DLL:
- kernel32.dll is present.
- oleaut32.dll is present.
- user32.dll is present.
PE_HEADER:
- Subsystem = 3.
- MajorOperatingSystemVersion = 6.
- MajorSubsystemVersion = 6.
PE_SECTION:


In [16]:
# ============================================================
# CELL 8C — FINAL ROBUST SAMPLE-SPECIFIC EVIDENCE
# ============================================================

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Load LOCKED training data
# ------------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/mohdtahasayed/malware-2-0/Malware_Train_60.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("=" * 80)
print("FINAL EVIDENCE EXTRACTION")
print("=" * 80)
print("Training shape:", train_df.shape)


# ------------------------------------------------------------
# 2. Feature groups
# ------------------------------------------------------------

api_features = [c for c in train_df.columns if c in api_features]
dll_features = [c for c in train_df.columns if c in dll_features]
header_features = [c for c in train_df.columns if c in pe_header_features]
section_features = [c for c in train_df.columns if c in pe_section_features]

feature_groups = {
    "API": api_features,
    "DLL": dll_features,
    "PE_HEADER": header_features,
    "PE_SECTION": section_features
}

print("\nFeature groups:")
for group, features in feature_groups.items():
    print(f"{group:12s}: {len(features)}")


# ------------------------------------------------------------
# 3. Class mapping
# ------------------------------------------------------------

CLASS_NAMES = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}


# ------------------------------------------------------------
# 4. Calculate class-vs-rest association
# ------------------------------------------------------------

y_train = train_df["Type"].astype(int)

class_counts = y_train.value_counts().to_dict()

class_effects = {}

for group, features in feature_groups.items():

    class_effects[group] = {}

    overall_std = (
        train_df[features]
        .std()
        .replace(0, np.nan)
    )

    for class_id in CLASS_NAMES:

        class_mask = y_train == class_id

        class_mean = (
            train_df.loc[class_mask, features]
            .mean()
        )

        rest_mean = (
            train_df.loc[~class_mask, features]
            .mean()
        )

        effect = (
            (class_mean - rest_mean)
            / overall_std
        )

        effect = (
            effect
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )

        class_effects[group][class_id] = effect


# ------------------------------------------------------------
# 5. Build percentile information
#
# Percentile is calculated using TRAINING data only.
# ------------------------------------------------------------

percentile_tables = {}

for group, features in feature_groups.items():

    percentile_tables[group] = {}

    for feature in features:

        values = train_df[feature].values

        sorted_values = np.sort(values)

        # Unique sorted values + cumulative empirical percentile
        unique_values, counts = np.unique(
            sorted_values,
            return_counts=True
        )

        cumulative = np.cumsum(counts)

        # Mid-rank percentile
        percentile = (
            cumulative - counts / 2
        ) / len(values)

        percentile_tables[group][feature] = (
            unique_values,
            percentile
        )


def get_percentile(group, feature, value):

    unique_values, percentiles = (
        percentile_tables[group][feature]
    )

    idx = np.searchsorted(
        unique_values,
        value,
        side="left"
    )

    if idx >= len(unique_values):
        idx = len(unique_values) - 1

    return float(percentiles[idx])


# ------------------------------------------------------------
# 6. Final evidence extractor
# ------------------------------------------------------------

def extract_sample_evidence(row, class_id, top_k=3):

    evidence = {}

    for group, features in feature_groups.items():

        effects = class_effects[group][class_id]

        # ====================================================
        # API / DLL
        # ====================================================

        if group in ["API", "DLL"]:

            present = [
                feature
                for feature in features
                if row[feature] == 1
            ]

            if not present:
                evidence[group] = []
                continue

            # Class-specific association of presence
            scores = effects.loc[present].abs()

            selected = (
                scores
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = [
                {
                    "feature": feature,
                    "value": 1
                }
                for feature in selected
            ]

        # ====================================================
        # PE Header / PE Section
        # ====================================================

        else:

            scores = {}

            for feature in features:

                value = float(row[feature])

                # Empirical percentile
                p = get_percentile(
                    group,
                    feature,
                    value
                )

                # Distance from median percentile.
                # 0 = typical
                # 1 = extremely unusual
                distinctiveness = abs(
                    p - 0.5
                ) * 2

                # Class-specific association
                class_strength = abs(
                    float(effects[feature])
                )

                # Combined score
                scores[feature] = (
                    class_strength *
                    distinctiveness
                )

            scores = pd.Series(scores)

            # ------------------------------------------------
            # Only retain features with some distinctiveness.
            # This prevents ordinary values from being treated
            # as strong evidence.
            # ------------------------------------------------

            scores = scores[
                scores > 0.05
            ]

            selected = (
                scores
                .sort_values(ascending=False)
                .head(top_k)
                .index
                .tolist()
            )

            evidence[group] = []

            for feature in selected:

                value = row[feature]

                if float(value).is_integer():
                    value = int(value)
                else:
                    value = float(value)

                evidence[group].append({
                    "feature": feature,
                    "value": value
                })

    return evidence


# ------------------------------------------------------------
# 7. Grounded evidence text
# ------------------------------------------------------------

def evidence_to_text(evidence):

    lines = []

    for group in [
        "API",
        "DLL",
        "PE_HEADER",
        "PE_SECTION"
    ]:

        items = evidence[group]

        if not items:
            continue

        lines.append(f"{group}:")

        for item in items:

            feature = item["feature"]
            value = item["value"]

            if group in ["API", "DLL"]:
                lines.append(
                    f"- {feature} is present."
                )
            else:
                lines.append(
                    f"- {feature} = {value}."
                )

    return "\n".join(lines)


# ------------------------------------------------------------
# 8. Generate 10 final prototype examples
# ------------------------------------------------------------

sample_df = (
    train_df
    .sample(
        n=10,
        random_state=42
    )
    .reset_index(drop=True)
)

prototype_records = []

for _, row in sample_df.iterrows():

    class_id = int(row["Type"])

    evidence = extract_sample_evidence(
        row,
        class_id,
        top_k=3
    )

    prototype_records.append({
        "id": row["SHA256"],
        "target": CLASS_NAMES[class_id],
        "target_id": class_id,
        "evidence": evidence,
        "evidence_text": evidence_to_text(evidence)
    })


# ------------------------------------------------------------
# 9. Display final evidence
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("10 FINAL EVIDENCE EXAMPLES")
print("=" * 80)

for i, record in enumerate(prototype_records, 1):

    print(
        f"\n{'=' * 30} SAMPLE {i} {'=' * 30}"
    )

    print("SHA256 :", record["id"])
    print("TARGET :", record["target"])

    print("\nEVIDENCE:")
    print(record["evidence_text"])


# ------------------------------------------------------------
# 10. Evidence statistics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EVIDENCE COUNT SUMMARY")
print("=" * 80)

for group in feature_groups:

    counts = [
        len(record["evidence"][group])
        for record in prototype_records
    ]

    print(
        f"{group:12s}: "
        f"mean={np.mean(counts):.2f}, "
        f"min={np.min(counts)}, "
        f"max={np.max(counts)}"
    )


# ------------------------------------------------------------
# 11. Save final prototype
# ------------------------------------------------------------

output_path = (
    "/kaggle/working/"
    "LLM_Evidence_Prototype_10_Final.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:

    for record in prototype_records:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print("\nSaved:")
print(output_path)

FINAL EVIDENCE EXTRACTION
Training shape: (17693, 320)

Feature groups:
API         : 200
DLL         : 27
PE_HEADER   : 46
PE_SECTION  : 45

10 FINAL EVIDENCE EXAMPLES

============================== SAMPLE 1 ==============================
SHA256 : 2df7b0d6ed9d2ad1dba3a9ba6b4d44bc4d7667b25b78b02b81cb0fa5b04fb9dd
TARGET : Benign

EVIDENCE:
API:
- exit is present.
- initterm is present.
- setusermatherr is present.
DLL:
- msvcrt.dll is present.
- kernel32.dll is present.
PE_HEADER:
- Magic = 523.
- SizeOfOptionalHeader = 240.
- Machine = 34404.
PE_SECTION:
- pdata_Characteristics = 1076887616.
- bss_Characteristics = 3228565632.
- idata_Characteristics = 3224371264.

============================== SAMPLE 2 ==============================
SHA256 : 2dd0bfc0d560531b164d18f09e870fdb9b4985527813d7358914218546f83b45
TARGET : BankingTrojan

EVIDENCE:
API:
- raiseexception is present.
- loadlibrarya is present.
- getprocaddress is present.
DLL:
- kernel32.dll is present.
- oleaut32.dll is presen

In [22]:
# ============================================================
# CELL 9 — Generate SFT Training JSONL
# ============================================================

import json
import os
from collections import Counter

OUTPUT_PATH = "/kaggle/working/LLM_SFT_Train_17693.jsonl"

# ------------------------------------------------------------
# 1. Class mapping
# ------------------------------------------------------------

CLASS_NAMES = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

# ------------------------------------------------------------
# 2. Confirm locked split
# ------------------------------------------------------------

print("Train:", train_df.shape)

assert train_df.shape[0] == 17693

# ------------------------------------------------------------
# 3. Exact feature groups from existing notebook state
# ------------------------------------------------------------

print("\nFeature groups:")
print("API:", len(api_features))
print("DLL:", len(dll_features))
print("PE Header:", len(pe_header_features))
print("PE Section:", len(pe_section_features))

assert len(api_features) == 200
assert len(dll_features) == 27
assert len(pe_header_features) == 46
assert len(pe_section_features) == 45

ALL_FEATURES = (
    api_features +
    dll_features +
    pe_header_features +
    pe_section_features
)

assert len(ALL_FEATURES) == 318
assert len(set(ALL_FEATURES)) == 318

print("Total ML features:", len(ALL_FEATURES))

# ------------------------------------------------------------
# 4. System prompt
# ------------------------------------------------------------

SYSTEM_PROMPT = (
    "You are a Windows PE malware classification assistant. "
    "Analyze only the provided static features. "
    "Do not infer runtime behavior that is not supported by the input."
)

# ------------------------------------------------------------
# 5. Build complete profile
# ------------------------------------------------------------

def build_llm_profile(row):

    # Only present API functions
    present_api = [
        feature
        for feature in api_features
        if row[feature] == 1
    ]

    # Only imported DLLs
    present_dll = [
        feature
        for feature in dll_features
        if row[feature] == 1
    ]

    # All PE Header values
    pe_header = {
        feature: (
            int(row[feature])
            if isinstance(row[feature], np.integer)
            else float(row[feature])
            if isinstance(row[feature], np.floating)
            else row[feature]
        )
        for feature in pe_header_features
    }

    # All PE Section values
    pe_section = {
        feature: (
            int(row[feature])
            if isinstance(row[feature], np.integer)
            else float(row[feature])
            if isinstance(row[feature], np.floating)
            else row[feature]
        )
        for feature in pe_section_features
    }

    return (
        present_api,
        present_dll,
        pe_header,
        pe_section
    )

# ------------------------------------------------------------
# 6. Build user prompt
# ------------------------------------------------------------

def build_user_prompt(row):

    (
        present_api,
        present_dll,
        pe_header,
        pe_section
    ) = build_llm_profile(row)

    return (
        "Analyze the following Windows PE static feature profile.\n\n"

        "[API FUNCTIONS PRESENT]\n"
        + json.dumps(
            present_api,
            ensure_ascii=False,
            separators=(",", ":")
        )

        + "\n\n[DLLS IMPORTED]\n"
        + json.dumps(
            present_dll,
            ensure_ascii=False,
            separators=(",", ":")
        )

        + "\n\n[PE HEADER]\n"
        + json.dumps(
            pe_header,
            ensure_ascii=False,
            separators=(",", ":")
        )

        + "\n\n[PE SECTIONS]\n"
        + json.dumps(
            pe_section,
            ensure_ascii=False,
            separators=(",", ":")
        )

        + "\n\nIdentify the malware family represented by this static profile."
        + "\nBase the assessment only on the provided static features."
        + "\nDo not assume runtime behavior that is not supported by the input."
    )

# ------------------------------------------------------------
# 7. Build assistant response
# ------------------------------------------------------------

def build_assistant_response(family, evidence):

    return (
        f"Predicted Malware Family: {family}\n\n"
        "Evidence:\n"
        + evidence_to_text(evidence)
    )

# ------------------------------------------------------------
# 8. Generate JSONL
# ------------------------------------------------------------

if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

class_counts = Counter()

print("\nGenerating SFT dataset...")

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    for i, (_, row) in enumerate(
        train_df.iterrows()
    ):

        class_id = int(row["Type"])

        family = CLASS_NAMES[class_id]

        # Existing Cell 8C evidence extractor
        evidence = extract_sample_evidence(
            row,
            class_id
        )

        record = {
            "id": str(row["SHA256"]),

            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": build_user_prompt(row)
                },
                {
                    "role": "assistant",
                    "content": build_assistant_response(
                        family,
                        evidence
                    )
                }
            ],

            "label": family,
            "label_id": class_id
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

        class_counts[class_id] += 1

        if (i + 1) % 2000 == 0:
            print(
                f"Generated {i + 1:,} / "
                f"{len(train_df):,}"
            )

print("\nGeneration complete.")

# ------------------------------------------------------------
# 9. Validate output
# ------------------------------------------------------------

jsonl_count = 0
jsonl_ids = set()
jsonl_class_counts = Counter()
first_record = None

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        record = json.loads(line)

        jsonl_count += 1

        if first_record is None:
            first_record = record

        record_id = record["id"]

        assert record_id not in jsonl_ids
        jsonl_ids.add(record_id)

        jsonl_class_counts[
            int(record["label_id"])
        ] += 1

        # Message structure
        assert len(record["messages"]) == 3

        assert record["messages"][0]["role"] == "system"
        assert record["messages"][1]["role"] == "user"
        assert record["messages"][2]["role"] == "assistant"

        user_text = record["messages"][1]["content"]

        # Required sections
        assert "[API FUNCTIONS PRESENT]" in user_text
        assert "[DLLS IMPORTED]" in user_text
        assert "[PE HEADER]" in user_text
        assert "[PE SECTIONS]" in user_text

        # SHA must NOT be given to model
        assert "SHA256" not in user_text

        # Correct output format
        assert record["messages"][2]["content"].startswith(
            "Predicted Malware Family:"
        )

# ------------------------------------------------------------
# 10. Check IDs and class counts
# ------------------------------------------------------------

train_ids = set(
    train_df["SHA256"].astype(str)
)

assert jsonl_count == 17693
assert len(jsonl_ids) == 17693
assert jsonl_ids == train_ids

expected_class_counts = Counter(
    train_df["Type"].astype(int)
)

assert jsonl_class_counts == expected_class_counts

# ------------------------------------------------------------
# 11. Check no contamination
# ------------------------------------------------------------

val_ids = set(
    val_df["SHA256"].astype(str)
)

test_ids = set(
    test_df["SHA256"].astype(str)
)

assert jsonl_ids.isdisjoint(val_ids)
assert jsonl_ids.isdisjoint(test_ids)

# ------------------------------------------------------------
# 12. Final statistics
# ------------------------------------------------------------

file_size_mb = (
    os.path.getsize(OUTPUT_PATH)
    / (1024 ** 2)
)

print("\n" + "=" * 65)
print("SFT DATASET VALIDATION")
print("=" * 65)

print(f"Records:    {jsonl_count:,}")
print(f"Unique IDs: {len(jsonl_ids):,}")
print(f"File size:  {file_size_mb:.2f} MB")

print("\nFeature groups:")
print(f"API:        {len(api_features)}")
print(f"DLL:        {len(dll_features)}")
print(f"PE Header:  {len(pe_header_features)}")
print(f"PE Section: {len(pe_section_features)}")
print(f"TOTAL:      {len(ALL_FEATURES)}")

print("\nClass distribution:")

for class_id in sorted(CLASS_NAMES):

    print(
        f"{class_id} | "
        f"{CLASS_NAMES[class_id]:<18} | "
        f"{jsonl_class_counts[class_id]:,}"
    )

print("\nContamination:")
print(
    "JSONL ∩ Validation:",
    len(jsonl_ids & val_ids)
)
print(
    "JSONL ∩ Test:",
    len(jsonl_ids & test_ids)
)

print("\nSaved:")
print(OUTPUT_PATH)

# ------------------------------------------------------------
# 13. Display first record
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FIRST GENERATED RECORD")
print("=" * 65)

print(
    json.dumps(
        first_record,
        indent=2,
        ensure_ascii=False
    )
)

Train: (17693, 320)

Feature groups:
API: 200
DLL: 27
PE Header: 46
PE Section: 45
Total ML features: 318

Generating SFT dataset...
Generated 2,000 / 17,693
Generated 4,000 / 17,693
Generated 6,000 / 17,693
Generated 8,000 / 17,693
Generated 10,000 / 17,693
Generated 12,000 / 17,693
Generated 14,000 / 17,693
Generated 16,000 / 17,693

Generation complete.

SFT DATASET VALIDATION
Records:    17,693
Unique IDs: 17,693
File size:  67.17 MB

Feature groups:
API:        200
DLL:        27
PE Header:  46
PE Section: 45
TOTAL:      318

Class distribution:
0 | Benign             | 1,127
1 | RedLineStealer     | 3,013
2 | Downloader         | 2,785
3 | RAT                | 2,973
4 | BankingTrojan      | 3,046
5 | SnakeKeyLogger     | 2,530
6 | Spyware            | 2,219

Contamination:
JSONL ∩ Validation: 0
JSONL ∩ Test: 0

Saved:
/kaggle/working/LLM_SFT_Train_17693.jsonl

FIRST GENERATED RECORD
{
  "id": "e942c853f791a2ebd37ae59344bf8878d9a02cdcb83848a68876c49497c642d2",
  "messages": [
    